# Artificial Intelligence – CSA17 | Assignment 1 (CO1)
## Complete Python 3 implementation for Tasks 1–12

**Execution environment:** Jupyter Notebook / Google Colab  
**Formatting note:** The written report should use Times New Roman, 12 pt, with bold headings. This notebook contains the executable source code and generated evidence.

### Experimental protocol and prediction (write this before running experiments)
**Prediction:** On the generated sparse binary-tree graphs, BFS and DFS with duplicate detection should expand no more than the number of reachable vertices. BFS should use a wider frontier than DFS on a deep, narrow traversal pattern. UCS should return the least-travel-time route but may perform more priority-queue work per expansion than BFS/DFS. As the number of vertices increases, operation counts should grow approximately linearly for BFS/DFS on this explicit graph family; UCS should show additional heap overhead. These are predictions, not measured results.

The benchmark uses fixed seeds and identical graph/start/goal inputs for all algorithms. It records preprocessing time separately. Run at least three graph sizes or seeds; the default experiment uses 1,000, 10,000 and 100,000 vertices. The optional 1,000,000-vertex run may require substantial RAM.

## Common model and data structures

The hospital is represented by a weighted undirected graph. A vertex is a junction/door/lift lobby; an edge is a traversable connection; edge weight is estimated travel time. BFS minimizes number of edges, DFS finds a route but is not optimal, and UCS minimizes total edge cost when costs are non-negative.

In [ ]:

from __future__ import annotations
from collections import deque
from dataclasses import dataclass
from typing import Optional, List, Tuple, Dict
import heapq, random, time, math, statistics, csv
from pathlib import Path

Adj = List[List[Tuple[int, float]]]

@dataclass
class SearchResult:
    algorithm: str
    path: Optional[List[int]]
    generated: int
    expanded: int
    max_frontier: int
    elapsed_s: float

    @property
    def depth(self):
        return None if self.path is None else len(self.path) - 1

def build_hospital_graph(n: int, seed: int = 42) -> Adj:
    """Reproducible sparse binary-tree graph with positive travel-time costs."""
    rng = random.Random(seed)
    graph = [[] for _ in range(n)]
    for child in range(1, n):
        parent = (child - 1) // 2
        w = float(rng.randint(1, 9))
        graph[parent].append((child, w))
        graph[child].append((parent, w))
    return graph

def reconstruct(parent, start, goal):
    path = [goal]
    while path[-1] != start:
        path.append(parent[path[-1]])
    return list(reversed(path))

def calculate_path_cost(graph, path):
    if path is None:
        return None
    total = 0.0
    for a, b in zip(path, path[1:]):
        total += next(w for v, w in graph[a] if v == b)
    return total

def bfs(graph: Adj, start: int, goal: int) -> SearchResult:
    t0 = time.perf_counter()
    if start == goal:
        return SearchResult("BFS", [start], 1, 0, 1, time.perf_counter()-t0)
    q = deque([start])
    visited = {start}
    parent = {}
    generated, expanded, max_frontier = 1, 0, 1
    found = False
    while q:
        v = q.popleft()
        expanded += 1
        for u, _ in graph[v]:
            if u not in visited:
                visited.add(u)
                parent[u] = v
                generated += 1
                if u == goal:
                    found = True
                    break
                q.append(u)
        max_frontier = max(max_frontier, len(q))
        if found:
            break
    path = reconstruct(parent, start, goal) if found else None
    return SearchResult("BFS", path, generated, expanded, max_frontier,
                        time.perf_counter()-t0)

def recursive_dfs(graph: Adj, start: int, goal: int,
                  depth_limit: Optional[int] = None) -> SearchResult:
    t0 = time.perf_counter()
    if start == goal:
        return SearchResult("Recursive DFS", [start], 1, 0, 1,
                            time.perf_counter()-t0)
    n = len(graph)
    limit = n - 1 if depth_limit is None else min(depth_limit, n - 1)
    visited = bytearray(n)
    parent = [-1] * n
    generated, expanded, max_stack = 1, 0, 1

    def visit(v, depth):
        nonlocal generated, expanded, max_stack
        if v == goal:
            return True
        if depth >= limit:
            return False
        visited[v] = 1
        expanded += 1
        max_stack = max(max_stack, depth + 1)
        for u, _ in graph[v]:
            if not visited[u]:
                visited[u] = 1
                parent[u] = v
                generated += 1
                if visit(u, depth + 1):
                    return True
        return False

    found = visit(start, 0)
    if found:
        path = [goal]
        while path[-1] != start:
            path.append(parent[path[-1]])
        path.reverse()
    else:
        path = None
    return SearchResult("Recursive DFS", path, generated, expanded, max_stack,
                        time.perf_counter()-t0)

def ucs(graph: Adj, start: int, goal: int) -> SearchResult:
    t0 = time.perf_counter()
    if start == goal:
        return SearchResult("UCS", [start], 1, 0, 1, time.perf_counter()-t0)
    heap = [(0.0, start)]
    best = {start: 0.0}
    parent = {}
    generated, expanded, max_frontier = 1, 0, 1
    found = False
    while heap:
        cost, v = heapq.heappop(heap)
        if cost != best.get(v):
            continue
        expanded += 1
        if v == goal:
            found = True
            break
        for u, w in graph[v]:
            new_cost = cost + w
            if new_cost < best.get(u, float("inf")):
                best[u] = new_cost
                parent[u] = v
                heapq.heappush(heap, (new_cost, u))
                generated += 1
        max_frontier = max(max_frontier, len(heap))
    path = reconstruct(parent, start, goal) if found else None
    return SearchResult("UCS", path, generated, expanded, max_frontier,
                        time.perf_counter()-t0)


## TASK 1 – Agent, PEAS, problem formulation and strategies

**PEAS:** Performance: correct delivery, low travel time, safety, reliability. Environment: corridors, junctions, lifts, wards, people. Actuators: wheels, steering, brakes, lift/door interface, compartment lock. Sensors: localization, camera/LiDAR/proximity, lift/door status.

**Environment:** partially observable in a real hospital; deterministic in the simplified known-graph model; sequential; dynamic; discrete graph abstraction (physical motion is continuous); multi-agent because people and other robots act in the same space. The agent follows the **acting-rationally** view: it chooses actions that maximize expected performance from its percepts and model.

**Problem:** state=current vertex; initial state=pharmacy; actions=move to an adjacent allowed vertex; transition=follow the selected edge; goal=current vertex equals requested ward; path cost=sum of edge travel times.

**History link:** classical symbolic AI represented problem solving as search through states using operators and goal tests (e.g., early General Problem Solver work).

The three distinct strategies are BFS (iterative queue), recursive depth-limited DFS, and UCS (iterative priority queue).

In [ ]:

# TASK 1 – Demonstrate all three strategies on the same small hospital map.
demo_graph = [
    [(1, 2.0), (2, 1.0)],  # pharmacy
    [(0, 2.0), (3, 2.0)],
    [(0, 1.0), (3, 8.0)],
    [(1, 2.0), (2, 8.0)]   # ward
]
for search in (bfs, recursive_dfs, ucs):
    result = search(demo_graph, 0, 3)
    print(search.__name__, "path=", result.path,
          "depth=", result.depth,
          "travel_time=", calculate_path_cost(demo_graph, result.path))
# Expected: BFS returns a 2-edge route; UCS selects 0->2->3 only if total cost is lower?
# Here 0->1->3 costs 4 and 0->2->3 costs 9, so UCS should choose 0->1->3.


## TASK 2 – Identify and count the basic operation

The basic operation for BFS/DFS is **successor examination** (checking an adjacency-list edge). For UCS, count **edge relaxations** and priority-queue pushes/pops. The following instrumented function counts adjacency entries examined for all three algorithms on one shared graph.

In [ ]:

def operation_counts(graph, start, goal):
    """Return comparable edge-examination counts and search results."""
    reports = {}
    # BFS edge examinations
    q, seen = deque([start]), {start}
    bfs_checks = 0
    while q:
        v = q.popleft()
        if v == goal: break
        for u, _ in graph[v]:
            bfs_checks += 1
            if u not in seen:
                seen.add(u); q.append(u)
    reports["BFS successor examinations"] = bfs_checks

    # DFS edge examinations
    seen = set()
    dfs_checks = 0
    def walk(v):
        nonlocal dfs_checks
        if v == goal: return True
        seen.add(v)
        for u, _ in graph[v]:
            dfs_checks += 1
            if u not in seen and walk(u): return True
        return False
    walk(start)
    reports["DFS successor examinations"] = dfs_checks

    # UCS edge relaxations and heap operations
    heap, best = [(0.0, start)], {start: 0.0}
    relaxations = pushes = pops = 0
    while heap:
        cost, v = heapq.heappop(heap); pops += 1
        if cost != best.get(v): continue
        if v == goal: break
        for u, w in graph[v]:
            relaxations += 1
            nc = cost + w
            if nc < best.get(u, float("inf")):
                best[u] = nc
                heapq.heappush(heap, (nc, u)); pushes += 1
    reports["UCS edge relaxations"] = relaxations
    reports["UCS heap pushes"] = pushes
    reports["UCS heap pops"] = pops
    return reports

small = build_hospital_graph(31, 10)
operation_counts(small, 0, 30)


## TASK 3 – Mathematical time and space functions

Let the implicit search tree have branching factor `b`, shallowest goal depth `d`, maximum depth `m`, optimal cost `C*`, and minimum positive edge cost `ε`.

At level `i`, there are `b^i` nodes. Therefore:
`S(d)=1+b+...+b^d=(b^(d+1)-1)/(b-1)` for `b != 1`.

- BFS tree model: time `Θ(b^d)`; frontier space `O(b^(d+1))`.
- Recursive DFS tree model: worst time `Θ(b^m)`; recursion stack `O(m)`.
- UCS tree model: any path of cost at most `C*` has at most `floor(C*/ε)` edges, giving a cost-depth bound `O(b^floor(C*/ε))`.
- Explicit finite graph with adjacency list and duplicate detection: BFS/DFS `O(V+E)` time, `O(V)` auxiliary memory; UCS with binary heap `O((V+E) log V)` time and `O(V)` auxiliary memory. Graph construction itself is `O(V+E)` time and space.

This cell evaluates the level-sum and graph-operation formulas numerically.

In [ ]:

def level_sum(b, depth):
    if b == 1:
        return depth + 1
    return (b**(depth + 1) - 1) // (b - 1)

for b, d, m in [(2, 5, 8), (3, 4, 7), (4, 3, 6)]:
    print({"b": b, "d": d, "m": m,
           "BFS_tree_nodes_through_d": level_sum(b, d),
           "DFS_tree_nodes_through_m": level_sum(b, m)})


## TASK 4 – Best, average and worst cases

Best case: `start == goal` (constant search work); for BFS/UCS, a nearby/cheap goal may also terminate early. Average case depends on goal placement, neighbour order and edge-cost distribution. Worst case: goal is last in the search order or unreachable; BFS/DFS can scan all reachable vertices/edges; UCS may process most vertices and heap entries before termination. The next cell checks the three required edge cases on the same graph.

In [ ]:

def validate_edge_cases():
    graph = build_hospital_graph(63, 19)
    isolated = graph + [[]]
    for fn in (bfs, recursive_dfs, ucs):
        reachable = fn(graph, 0, 62)
        same = fn(graph, 7, 7)
        unreachable = fn(isolated, 0, 63)
        assert reachable.path is not None and reachable.path[0] == 0 and reachable.path[-1] == 62
        assert same.path == [7]
        assert unreachable.path is None
        print(fn.__name__, {
            "reachable": reachable.path is not None,
            "start_equals_goal": same.path,
            "unreachable_returns_none": unreachable.path is None
        })
validate_edge_cases()


In [ ]:

# TASK 4 – Empirical case behaviour: vary the goal's location in the same graph.
case_graph = build_hospital_graph(255, 31)
case_goals = {"best_near_start": 1, "middle": 127, "deep_last": 254}
for label, target in case_goals.items():
    print("\nCase:", label, "goal:", target)
    for fn in (bfs, recursive_dfs, ucs):
        r = fn(case_graph, 0, target)
        print(fn.__name__, "generated=", r.generated,
              "expanded=", r.expanded, "max_frontier=", r.max_frontier,
              "depth=", r.depth)


## TASK 5 – Asymptotic analysis

For an implicit tree, BFS is `Θ(b^d)` time and `O(b^(d+1))` frontier space; depth-limited recursive DFS is `Θ(b^m)` time and `O(m)` stack space; UCS has a cost-bound exponential tree expression in `floor(C*/ε)`. For the explicit finite graph with duplicate detection, BFS/DFS are `O(V+E)` and UCS is `O((V+E)log V)`. At one million locations, map storage and visited/parent arrays matter for every algorithm; BFS may have a large frontier; recursive DFS may hit recursion limits on long paths; UCS has heap overhead but correctly minimizes weighted travel time.

In [ ]:

# TASK 5 – Evaluate theoretical graph-operation growth at the assignment sizes.
for V in (1_000, 10_000, 100_000, 1_000_000):
    E = V - 1  # connected tree
    print(f"V={V:,} E={E:,} | BFS/DFS edge-scan upper bound={2*E:,} "
          f"| UCS heap scale proxy={(V+E)*math.log2(V):,.0f}")


## TASK 6 – Derive recurrence for recursive DFS

Let `T(l)` be worst-case work with `l` levels remaining. At `l=0`, only base-case tests execute: `T(0)=c0`. At `l>0`, local work is `c` and at most `b` recursive calls each receive `l-1`: `T(l)=bT(l-1)+c`. The next cell evaluates the recurrence exactly for sample parameters.

In [ ]:

def recurrence_value(b, depth, c=1, base=1):
    """Compute T(0)=base; T(l)=b*T(l-1)+c."""
    T = base
    values = [T]
    for _ in range(depth):
        T = b*T + c
        values.append(T)
    return values

for b in (1, 2, 3):
    print("b =", b, "T(0..6) =", recurrence_value(b, 6))


## TASK 7 – Solve recurrence by substitution

Repeated substitution gives:
`T(l)=b^l T(0) + c(1+b+...+b^(l-1))`
`= b^l c0 + c(b^l-1)/(b-1)` for `b != 1`.

Thus `T(l)=Θ(b^l)` for `b>1`, and `Θ(l)` for `b=1`. Iterative deepening repeats depth-limited DFS for limits `0..d`; its total work is `Σ O(b^i)=Θ(b^d)` for `b>1`, with `O(d)` stack space. The following cell compares the recurrence value with its closed form.

In [ ]:

def recurrence_closed_form(b, depth, c=1, base=1):
    if b == 1:
        return base + c*depth
    return (b**depth)*base + c*((b**depth - 1)//(b-1))

for b in (2, 3, 4):
    vals = recurrence_value(b, 8, c=2, base=3)
    closed = recurrence_closed_form(b, 8, c=2, base=3)
    assert vals[-1] == closed
    print(f"b={b}: iterative recurrence={vals[-1]}, closed form={closed}")


## TASK 8 – Implement and test all search algorithms

The algorithms are implemented above. This cell uses identical graphs, start and goal pairs and fixed seeds. It verifies route validity, reachability, unreachable-goal behaviour and the start-equals-goal case.

In [ ]:

def check_route(graph, result, start, goal):
    if result.path is None:
        return False
    if result.path[0] != start or result.path[-1] != goal:
        return False
    for a, b in zip(result.path, result.path[1:]):
        if not any(v == b for v, _ in graph[a]):
            return False
    return True

def task8_tests():
    graph = build_hospital_graph(127, 2026)
    graph_unreachable = graph + [[]]
    for fn in (bfs, recursive_dfs, ucs):
        r = fn(graph, 0, 126)
        assert check_route(graph, r, 0, 126)
        assert fn(graph, 9, 9).path == [9]
        assert fn(graph_unreachable, 0, 127).path is None
        print(fn.__name__, "reachable route valid; start=goal valid; unreachable correctly reported")
task8_tests()


## TASK 9 – Experimental performance (at least three input sizes/seeds)

**Metrics:** generated nodes, expanded nodes, maximum frontier, solution depth, path cost, preprocessing time and mean search time over at least five runs. The CSV is the raw evidence; plots use logarithmic axes. The benchmark includes three sizes by default and can be expanded to one million vertices. Execution time is secondary to operation counts.

In [ ]:

def benchmark(sizes=(1_000, 10_000, 100_000), seeds=(11, 22, 33), runs=5,
              output_csv="task9_performance.csv"):
    records = []
    for n in sizes:
        for seed in seeds:
            t0 = time.perf_counter()
            graph = build_hospital_graph(n, seed)
            preprocess = time.perf_counter() - t0
            start, goal = 0, n-1
            for fn in (bfs, recursive_dfs, ucs):
                results = [fn(graph, start, goal) for _ in range(runs)]
                first = results[0]
                record = {
                    "vertices": n, "edges": n-1, "seed": seed,
                    "algorithm": first.algorithm, "runs": runs,
                    "preprocess_s": preprocess,
                    "mean_search_s": statistics.mean(x.elapsed_s for x in results),
                    "generated": first.generated, "expanded": first.expanded,
                    "max_frontier": first.max_frontier,
                    "solution_depth": first.depth,
                    "path_cost": calculate_path_cost(graph, first.path),
                    "reachable": first.path is not None
                }
                records.append(record)
                print(record)
    with open(output_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0].keys()))
        writer.writeheader(); writer.writerows(records)
    return records

# Run this cell to generate the actual evidence. Increase sizes to include 1,000,000.
records = benchmark()


### Figures and tables for Task 9

Run this cell after the benchmark. It creates a table and three sequentially captioned figures. Refer to these captions in the written discussion.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(records)
display(df)
for metric, ylabel, fig_no, filename in [
    ("mean_search_s", "Mean search time (seconds)", 1, "Figure_1_Search_Time.png"),
    ("expanded", "Nodes expanded", 2, "Figure_2_Nodes_Expanded.png"),
    ("max_frontier", "Maximum frontier size", 3, "Figure_3_Max_Frontier.png")
]:
    plt.figure(figsize=(7, 4))
    for algorithm, group in df.groupby("algorithm"):
        summary = group.groupby("vertices")[metric].mean().sort_index()
        plt.plot(summary.index, summary.values, marker="o", label=algorithm)
    plt.xscale("log"); plt.yscale("log")
    plt.xlabel("Number of vertices (log scale)")
    plt.ylabel(ylabel); plt.title(f"Figure {fig_no}. {ylabel} vs hospital graph size")
    plt.legend(); plt.grid(True, which="both", alpha=.25); plt.tight_layout()
    plt.savefig(filename, dpi=160)
    plt.show()


## TASK 10 – Compare theoretical and experimental behaviour

Interpret the actual CSV and plots; do not invent results. For the generated tree, BFS and DFS edge examinations should be bounded by the reachable graph's adjacency entries. BFS's frontier can be broad; DFS's call stack follows the current branch. UCS uses heap operations and may have higher time per expansion. Compare growth across sizes and seeds. Differences can arise from graph topology, neighbour ordering, edge weights, Python overhead, memory pressure and OS scheduling.

In [ ]:

# TASK 10 – Produce a compact comparison table from measured results.
comparison = (df.groupby("algorithm")
              .agg(mean_generated=("generated", "mean"),
                   mean_expanded=("expanded", "mean"),
                   mean_frontier=("max_frontier", "mean"),
                   mean_time_s=("mean_search_s", "mean"),
                   max_vertices=("vertices", "max"))
              .reset_index())
display(comparison)
print("Interpretation guide:")
print("- Compare expanded nodes first; these are less noisy than wall-clock time.")
print("- Compare maximum frontier as a memory proxy.")
print("- UCS heap overhead is expected because it orders by cumulative path cost.")


## TASK 11 – Final design recommendation

Recommend UCS when the objective is minimum total travel time and all edge costs are non-negative. BFS optimizes edge count, not weighted travel time; DFS is not optimal and depends on branch order. Reuse the adjacency-list map across deliveries to amortize preprocessing. For large maps, monitor heap and memory use. If costs or obstacles change, update the graph and replan.

In [ ]:

# TASK 11 – Verify UCS optimality on a small weighted graph by comparison
weighted_map = [
    [(1, 2.0), (2, 1.0)],
    [(0, 2.0), (3, 2.0)],
    [(0, 1.0), (3, 8.0)],
    [(1, 2.0), (2, 8.0)]
]
b = bfs(weighted_map, 0, 3)
u = ucs(weighted_map, 0, 3)
print("BFS route/cost:", b.path, calculate_path_cost(weighted_map, b.path))
print("UCS route/cost:", u.path, calculate_path_cost(weighted_map, u.path))
assert calculate_path_cost(weighted_map, u.path) <= calculate_path_cost(weighted_map, b.path)


## TASK 12 – Day-4 localization failure: belief-state experiment

The robot no longer knows one exact vertex. Its state becomes a **belief state** `B`, a set of possible locations. For `k` candidate locations, the theoretical number of subsets is at most `2^k`; for k=2,4,8 this is 4,16,256. A perfectly identifying local sensor maps the current belief to one singleton, leaving k possible singleton outcomes. Without sensing, an action must be safe/valid from every possible current location; with sensing, use an observation-conditioned policy.

The next cell is an explicitly simplified line-world experiment. It measures belief-set counts and compares worst-case move length. It assumes a perfect sensor and known line topology; a real deployment must model sensor error and observation outcomes.

In [ ]:

from itertools import combinations

def all_belief_states(k):
    items = list(range(k))
    states = []
    for r in range(k+1):
        states.extend(frozenset(x) for x in combinations(items, r))
    return states

def line_world_belief_demo(k):
    # Line positions 0..k-1; docking station is 0; candidates equally likely.
    candidates = list(range(k))
    total_beliefs = len(all_belief_states(k))
    # Without localization, one fixed robust plan must work from the farthest candidate.
    robust_moves_without_sensor = max(candidates)
    # With a perfect sensor, the robot learns its exact position and chooses a
    # conditional route. Expected moves are the mean distance across candidates.
    conditional_moves = [position for position in candidates]
    expected_moves_after_sensor = sum(conditional_moves) / k
    worst_moves_after_sensor = max(conditional_moves)
    return {
        "k": k,
        "possible_belief_subsets": total_beliefs,
        "singleton_beliefs_after_perfect_sensor": k,
        "robust_moves_without_sensor_worst_case": robust_moves_without_sensor,
        "after_sensor_expected_moves": expected_moves_after_sensor,
        "after_sensor_worst_case_moves": worst_moves_after_sensor,
        "sensor_actions": 1
    }

belief_results = [line_world_belief_demo(k) for k in (2, 4, 8)]
pd.DataFrame(belief_results)


### Task 12 interpretation and design decision

**Decision: redesign the uncertain-start layer, retain graph search underneath.** Do not select one arbitrary candidate location and run ordinary UCS: that may fail from the robot's actual location. Represent uncertainty as a belief set, use local sensing to reduce the set, then execute a conditional plan based on the observation. Once localized, use UCS to reach the pharmacy docking station. The simple line-world experiment demonstrates the subset-state growth and the benefit of a perfect localizing observation; it is not a complete probabilistic sensor model.

In [ ]:

# Final reproducibility record
print("Python version:", __import__("sys").version)
print("Random seeds used in benchmark:", (11, 22, 33))
print("Benchmark sizes:", (1_000, 10_000, 100_000))
print("Runs per size/seed/algorithm:", 5)
print("CSV:", "task9_performance.csv")
print("Figures:", "Figure_1_Search_Time.png, Figure_2_Nodes_Expanded.png, Figure_3_Max_Frontier.png")
